# Drone Security Analyst Agent Notebook
This notebook demonstrates how to run and interact with the drone security analyst agent prototype.


In [ ]:
# Install required packages (if not already installed)
%pip install fastapi uvicorn pydantic


In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# Show that the app starts
print('TestClient created')


In [ ]:
# Trigger background task manually for demo
import asyncio
from app.main import stream_drone_pipeline

# Run the pipeline for a short time to generate some data
async def run_demo():
    # We'll run a couple iterations
    for i, packet in enumerate([{'timestamp':'2026-08-18T00:01:00','lat':18.5204,'lon':73.8567,'alt':15.0,'location':'Main Gate','frame_desc':'A lone individual in a dark hoodie pacing back and forth near the fence line.'}, {'timestamp':'2026-08-18T06:15:00','lat':18.5206,'lon':73.8569,'alt':20.0,'location':'North Perimeter','frame_desc':'Clear field of view, deer grazing near the tree line.'}]):
        await asyncio.sleep(0.1)  # short delay
        telemetry_ctx = {'timestamp':packet['timestamp'],'lat':packet['lat'],'lon':packet['lon'],'alt':packet['alt'],'location':packet['location']}
        from app.main import indexed_database, security_alerts
        db_entry = {'id':len(indexed_database)+1,'timestamp':packet['timestamp'],'location':packet['location'],'telemetry':telemetry_ctx,'description':packet['frame_desc']}
        indexed_database.append(db_entry)
        print(f'Frame Indexed: ID {db_entry["id"]} at Location: {db_entry["location"]}')
        # simple alert check
        if 'person' in packet['frame_desc'].lower() and ('hoodie' in packet['frame_desc'].lower()):
            alert = {'type':'CRITICAL_ALERT','message':f"Security Breached: {packet['frame_desc']} at {packet['location']}",'timestamp':packet['timestamp'],'severity':'High'}
            security_alerts.append(alert)
            print(f'🚨 ALERT GENERATED: {alert["message"]}')
    
asyncio.run(run_demo())


In [ ]:
# Query logs
response = client.get('/logs')
print('Logs response:', response.json())


In [ ]:
# Query alerts
response = client.get('/alerts')
print('Alerts response:', response.json())


In [ ]:
# Search for 'truck'
response = client.post('/search', json={"query": "truck"})
print('Search response:', response.json())


In [ ]:
# Bonus chat
response = client.post('/chat-bonus', json={"query": "Tell me about the truck sightings."})
print('Chat bonus response:', response.json())
